In [1]:
%pip install beautifulsoup4
%pip install contractions
%pip install joblib
%pip install kaleido
%pip install langdetect
%pip install matplotlib
%pip install nltk
%pip install numpy
%pip install optuna
%pip install pandas
%pip install plotly
%pip install scikit-learn
%pip install seaborn
%pip install spacy
%pip install spacymoji
%pip install wordcloud
%pip install imbalanced-learn


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you 

In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pickle
import re
import string
import sys
import time
import unicodedata
from collections import Counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import spacymoji
from mpl_toolkits.mplot3d import Axes3D
from spacy import displacy
from spacy.lang.fr import French
from spacy.tokenizer import _get_regex_pattern
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objs as go
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as py
from plotly.subplots import make_subplots
from scipy.sparse import issparse

import nltk
from nltk import pos_tag, sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.stem.snowball import SnowballStemmer
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("tagsets")
nltk.download("tagsets_json")
nltk.download("wordnet")

import sklearn
from sklearn import metrics
from sklearn.base import BaseEstimator, TransformerMixin , ClassifierMixin
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

import optuna
import contractions
from bs4 import BeautifulSoup
from langdetect import detect

!python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/mario/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/mario/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /home/mario/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/mario/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/mario/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package tagsets to /home/mario/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!
[nltk_data] Downloading package tagsets_json to
[nltk_data]     /home/mario/nltk_data...
[nltk_data]   Package tagse

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 58.8 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## 🚀 Run from here

In [3]:
df=pd.read_csv('scitweets_export.tsv',sep='\t')
print (df.head())

   Unnamed: 0            tweet_id  \
0           0  316669998137483264   
1           1  319090866545385472   
2           2  322030931022065664   
3           3  322694830620807168   
4           4  328524426658328576   

                                                text  science_related  \
0  Knees are a bit sore. i guess that's a sign th...                0   
1          McDonald's breakfast stop then the gym 🏀💪                0   
2  Can any Gynecologist with Cancer Experience ex...                1   
3  Couch-lock highs lead to sleeping in the couch...                1   
4  Does daily routine help prevent problems with ...                1   

   scientific_claim  scientific_reference  scientific_context  
0               0.0                   0.0                 0.0  
1               0.0                   0.0                 0.0  
2               1.0                   0.0                 0.0  
3               1.0                   0.0                 0.0  
4               1.

In [4]:
def normalize(X,
              removelowercase=False,
              removestopwords=False,
              removedigit=False,
              removeemoji=False,
              getstemmer=False,
              getlemmatisation=False):
    
    nlp = spacy.load("en_core_web_sm")
    nlp.add_pipe("emoji", first=True)
    re_token_match = spacy.tokenizer._get_regex_pattern(nlp.Defaults.token_match)
    re_token_match = f"({re_token_match}|#\\w+|\\w+-\\w+)"
    X = contractions.fix(X)
    
    if removelowercase:
        X = X.lower()
    
    doc = nlp(X)
    
    lem = WordNetLemmatizer()
    stemmer = PorterStemmer()
    
    tokens = []
    
    for t in doc:
        if t.is_punct or t.is_space:
            continue
        if removestopwords and t.is_stop:
            continue
        if removedigit and t.like_num:
            continue
        if removeemoji and t._.is_emoji:
            continue
        
        word = t.text
        if getlemmatisation:
            word = lem.lemmatize(word)
        if getstemmer:
            word = stemmer.stem(word)
            
        tokens.append(word)
    
    return " ".join(tokens)   

In [5]:
class TextNormalizer(BaseEstimator, TransformerMixin):
    def __init__(self,
              removelowercase=True,
              removestopwords=True,
              removedigit=True,
              removeemoji=True,
              getstemmer=True,
              getlemmatisation=True):
        
        self.removelowercase = removelowercase
        self.removestopwords = removestopwords
        self.removedigit = removedigit
        self.removeemoji = removeemoji
        self.getstemmer = getstemmer
        self.getlemmatisation = getlemmatisation
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        return [
            normalize(text,
            removelowercase=self.removelowercase,
            removestopwords=self.removestopwords,
            removedigit=self.removedigit,
            removeemoji=self.removeemoji,
            getstemmer=self.getstemmer,
            getlemmatisation=self.getlemmatisation)
            for text in X
        ]

In [6]:

class CoinTossClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, random_state=None):
        self.random_state = random_state
    
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.n_classes_ = len(self.classes_)
        return self
    
    def predict(self, X):
        n_samples = X.shape[0] if issparse(X) else len(X)
        rng = np.random.RandomState(self.random_state)
        return rng.choice(self.classes_, size=n_samples)
    
    def predict_proba(self, X):
        n_samples = X.shape[0] if issparse(X) else len(X)
        rng = np.random.RandomState(self.random_state)
        proba = rng.dirichlet(np.ones(self.n_classes_), n_samples)
        return proba

In [7]:
def _build_pipeline(clf_name, clf, sampling, normalizer_params=None, vectorizer_params=None):
    norm_params = normalizer_params or {}
    vec_params = vectorizer_params or {}
    steps = [
        ("cleaner", TextNormalizer(**norm_params)),
        ("vectorizer", TfidfVectorizer(**vec_params)),
    ]
    if sampling == "oversample":
        steps.append(("sampler", SMOTE(random_state=42)))
    steps.append((clf_name, clf))
    return ImbPipeline(steps)


_NGRAM_MAP_WORD = {"1_1": (1, 1), "1_2": (1, 2), "1_3": (1, 3)}


def _suggest_normalizer_params(trial):
    norm_strategy = trial.suggest_categorical("norm_strategy", ["none", "stem", "lemma"])
    return {
        "removelowercase": trial.suggest_categorical("norm_lowercase", [True, False]),
        "removestopwords": trial.suggest_categorical("norm_stopwords", [True, False]),
        "removedigit": trial.suggest_categorical("norm_digit", [True, False]),
        "removeemoji": trial.suggest_categorical("norm_emoji", [True, False]),
        "getstemmer": (norm_strategy == "stem"),
        "getlemmatisation": (norm_strategy == "lemma"),
    }


def _suggest_vectorizer_params(trial):
    analyzer = trial.suggest_categorical("vec_analyzer", ["word", "char_wb"])
    if analyzer == "word":
        ngram_range = _NGRAM_MAP_WORD[trial.suggest_categorical("vec_ngram_word", ["1_1", "1_2", "1_3"])]
    else:
        nmin = trial.suggest_int("vec_ngram_char_min", 2, 4)
        nmax = trial.suggest_int("vec_ngram_char_max", 4, 6)
        if nmin > nmax:
            nmax = nmin
        ngram_range = (nmin, nmax)
    return {
        "analyzer": analyzer,
        "ngram_range": ngram_range,
        "min_df": trial.suggest_int("vec_min_df", 1, 5),
        "max_df": trial.suggest_float("vec_max_df", 0.7, 1.0),
        "sublinear_tf": trial.suggest_categorical("vec_sublinear_tf", [True, False]),
    }


def _normalizer_params_from_best(best_params):
    norm_strategy = best_params.get("norm_strategy", "none")
    return {
        "removelowercase": best_params.get("norm_lowercase", False),
        "removestopwords": best_params.get("norm_stopwords", False),
        "removedigit": best_params.get("norm_digit", False),
        "removeemoji": best_params.get("norm_emoji", False),
        "getstemmer": (norm_strategy == "stem"),
        "getlemmatisation": (norm_strategy == "lemma"),
    }


def _vectorizer_params_from_best(best_params):
    analyzer = best_params.get("vec_analyzer", "word")
    if analyzer == "word":
        ngram_range = _NGRAM_MAP_WORD[best_params.get("vec_ngram_word", "1_1")]
    else:
        nmin = best_params.get("vec_ngram_char_min", 3)
        nmax = best_params.get("vec_ngram_char_max", 5)
        if nmin > nmax:
            nmax = nmin
        ngram_range = (nmin, nmax)
    return {
        "analyzer": analyzer,
        "ngram_range": ngram_range,
        "min_df": best_params.get("vec_min_df", 1),
        "max_df": best_params.get("vec_max_df", 1.0),
        "sublinear_tf": best_params.get("vec_sublinear_tf", False),
    }


def objective(trial, X, y):
    model_name = trial.suggest_categorical(
        "model_name",
        ["KNeighbors", "DecisionTree", "SVC", "RandomForest", "CoinToss"]
    )

    sampling = trial.suggest_categorical("sampling", ["none", "oversample"])

    normalizer_params = _suggest_normalizer_params(trial)
    vectorizer_params = _suggest_vectorizer_params(trial)

    if model_name == "DecisionTree":
        max_depth = trial.suggest_int("max_depth", 1, 10)
        criterion = trial.suggest_categorical("criterion", ["gini", "entropy"])
        model = _build_pipeline(model_name, DecisionTreeClassifier(
            max_depth=max_depth,
            criterion=criterion,
            class_weight='balanced'
        ), sampling, normalizer_params, vectorizer_params)

    elif model_name == "KNeighbors":
        n_neighbors = trial.suggest_int("n_neighbors", 1, 15)
        weights = trial.suggest_categorical("weights", ["uniform", "distance"])
        p = trial.suggest_categorical("p", [1, 2])
        model = _build_pipeline(model_name, KNeighborsClassifier(
            n_neighbors=n_neighbors,
            weights=weights,
            p=p
        ), sampling, normalizer_params, vectorizer_params)

    elif model_name == "SVC":
        C = trial.suggest_float("C", 0.1, 100, log=True)
        kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])
        gamma = trial.suggest_categorical("gamma", ["scale", "auto"])
        model = _build_pipeline(model_name, SVC(
            C=C,
            kernel=kernel,
            gamma=gamma,
            probability=True,
            decision_function_shape='ovr',
            class_weight='balanced'
        ), sampling, normalizer_params, vectorizer_params)

    elif model_name == "RandomForest":
        n_estimators = trial.suggest_int("n_estimators", 10, 200)
        max_depth = trial.suggest_int("max_depth", 2, 20)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)
        model = _build_pipeline(model_name, RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            class_weight='balanced',
            n_jobs=-1
        ), sampling, normalizer_params, vectorizer_params)

    else:  # "CoinToss"
        model = _build_pipeline(model_name, CoinTossClassifier(random_state=42),
                                sampling, normalizer_params, vectorizer_params)

    score = cross_val_score(
        model, X, y,
        scoring="balanced_accuracy",
        n_jobs=-1
    ).mean()

    return score


def findBestModel(X, y):
    def objective_with_data(trial):
        return objective(trial, X, y)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective_with_data, n_trials=40)  # 40 de base

    print("Meilleur score :", round(study.best_value, 3))
    print("Meilleurs paramètres :", study.best_params)

    best_params = study.best_params
    model_name = best_params["model_name"]
    sampling = best_params.get("sampling", "none")
    normalizer_params = _normalizer_params_from_best(best_params)
    vectorizer_params = _vectorizer_params_from_best(best_params)

    if model_name == "DecisionTree":
        best_model = _build_pipeline(model_name, DecisionTreeClassifier(
            max_depth=best_params["max_depth"],
            criterion=best_params["criterion"],
            class_weight='balanced'
        ), sampling, normalizer_params, vectorizer_params)

    elif model_name == "KNeighbors":
        best_model = _build_pipeline(model_name, KNeighborsClassifier(
            n_neighbors=best_params["n_neighbors"],
            weights=best_params["weights"],
            p=best_params["p"]
        ), sampling, normalizer_params, vectorizer_params)

    elif model_name == "SVC":
        best_model = _build_pipeline(model_name, SVC(
            C=best_params["C"],
            kernel=best_params["kernel"],
            gamma=best_params["gamma"],
            probability=True,
            decision_function_shape='ovr',
            class_weight='balanced'
        ), sampling, normalizer_params, vectorizer_params)

    elif model_name == "RandomForest":
        best_model = _build_pipeline(model_name, RandomForestClassifier(
            n_estimators=best_params["n_estimators"],
            max_depth=best_params["max_depth"],
            min_samples_split=best_params["min_samples_split"],
            min_samples_leaf=best_params["min_samples_leaf"],
            class_weight='balanced',
            n_jobs=-1
        ), sampling, normalizer_params, vectorizer_params)

    else:  # "CoinToss"
        best_model = _build_pipeline(model_name, CoinTossClassifier(random_state=42),
                                     sampling, normalizer_params, vectorizer_params)

    return (model_name, best_model)


X = df.text
y = df.science_related

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

best_model = findBestModel(X_train, y_train)
best_model[1].fit(X_train, y_train)

# Évaluation sur jeu de test indépendant.
print("\n=== Évaluation sur jeu de test indépendant (Stage 1) ===")
y_pred = best_model[1].predict(X_test)
print(classification_report(y_test, y_pred, target_names=["NON-SCI", "SCI"]))
print(f"Balanced accuracy (held-out): {metrics.balanced_accuracy_score(y_test, y_pred):.3f}")

# Re-fit final sur l'intégralité du corpus pour la cascade en aval.
best_model[1].fit(X, y)
print("Le nom du meilleur model : " + best_model[0])
print("Apprentissage final terminé sur l'ensemble du corpus.")

model_path = "BestScientific_" + best_model[0] + ".pkl"
with open(model_path, "wb") as f:
    pickle.dump(best_model[1], f)

[I 2026-05-02 19:43:47,699] A new study created in memory with name: no-name-db6c4d6b-54b9-4491-863a-4cc7b457c04e
[I 2026-05-02 19:47:45,340] Trial 0 finished with value: 0.721515393842463 and parameters: {'model_name': 'DecisionTree', 'sampling': 'none', 'norm_strategy': 'stem', 'norm_lowercase': True, 'norm_stopwords': True, 'norm_digit': True, 'norm_emoji': True, 'vec_analyzer': 'word', 'vec_ngram_word': '1_3', 'vec_min_df': 1, 'vec_max_df': 0.7750086083087113, 'vec_sublinear_tf': True, 'max_depth': 5, 'criterion': 'entropy'}. Best is trial 0 with value: 0.721515393842463.
[I 2026-05-02 19:51:45,026] Trial 1 finished with value: 0.7723217379714781 and parameters: {'model_name': 'SVC', 'sampling': 'none', 'norm_strategy': 'none', 'norm_lowercase': False, 'norm_stopwords': True, 'norm_digit': True, 'norm_emoji': True, 'vec_analyzer': 'word', 'vec_ngram_word': '1_2', 'vec_min_df': 5, 'vec_max_df': 0.8722574253834593, 'vec_sublinear_tf': True, 'C': 0.16286608663770077, 'kernel': 'linear

Meilleur score : 0.795
Meilleurs paramètres : {'model_name': 'SVC', 'sampling': 'none', 'norm_strategy': 'none', 'norm_lowercase': False, 'norm_stopwords': True, 'norm_digit': True, 'norm_emoji': True, 'vec_analyzer': 'word', 'vec_ngram_word': '1_2', 'vec_min_df': 2, 'vec_max_df': 0.8641031587503901, 'vec_sublinear_tf': True, 'C': 0.40261770830202875, 'kernel': 'linear', 'gamma': 'auto'}

=== Évaluation sur jeu de test indépendant (Stage 1) ===
              precision    recall  f1-score   support

     NON-SCI       0.92      0.80      0.86       153
         SCI       0.68      0.87      0.76        75

    accuracy                           0.82       228
   macro avg       0.80      0.83      0.81       228
weighted avg       0.84      0.82      0.82       228

Balanced accuracy (held-out): 0.832
Le nom du meilleur model : SVC
Apprentissage final terminé sur l'ensemble du corpus.


In [8]:
# Tweets de test :
tab = ["Covid is chinese made","Albatross is the best bird","Buy Loreal for cheap","the earth is flat","Moon landing was fake","the space doesn't exist"]

# Tâche 1 : SCI vs NON-SCI                                                                                                                                                    
tweets_task1 = [                                                       
      "Poverty is the greatest cause behind child labour.",                                                                                                                                   
      "Does daily routine help prevent bipolar disorder?",
      "Lupus Research Institute Awards $1M Grants to Discover What Causes Lupus",                                                                                                             
      "The House science committee is now demanding information on grants.",                                                                                                                  
      "McDonald's breakfast stop then the gym 🏀💪",
      "Knees are a bit sore, I guess my treadmilling is working",                                                                                                                             
      "@HankAzaria oh loneliness and cheeseburgers are a dangerous mix",                                                                                                                      
      "Liverpool are a left back and a right wing away from the CL.",
  ]                                                                                                                                                                                           
                                                                         
# Tâche 2 : {CLAIM,REF} vs {CONTEXT}
tweets_task2 = [
      "Poverty is the greatest cause behind child labour.",                                                                                                                                   
      "Does daily routine help prevent bipolar disorder?",
      "Couch-lock highs lead to sleeping in the couch.",                                                                                                                                      
      "Can playing games improve lives? via @PostArcade",                
      "#MSGDonated4Million to support the research of Thalassemia & AIDS cure",
      "pls support #NIH funding to help fight #pancreaticcancer @PanCAN",                                                                                                                     
      "The who, what, when, where of the @BloombergCities #Iteams research",
      "Lupus Research Institute Awards $1M Grants to Discover What Causes Lupus",                                                                                                             
  ]                                                                      
                                                                                                                                                                                              
# Tâche 3 : {CLAIM} vs {REF} vs {CONTEXT}
tweets_task3 = [                       
      "Poverty is the greatest cause behind child labour.",
      "Couch-lock highs lead to sleeping in the couch.",                                                                                                                                      
      "Does daily routine help prevent bipolar disorder? http://t.co/...",
      "Can playing games improve lives? via @PostArcade http://t.co/...",                                                                                                                     
      "Study shows vaccines reduce hospitalizations — see @NatureMedicine",                                                                                                                   
      "#MSGDonated4Million to support Thalassemia & AIDS research",
      "pls support #NIH funding to fight #pancreaticcancer @PanCAN",                                                                                                                          
      "The House science committee demands info on NSF grants. http://t.co/...",                                                                                                              
  ] 

In [9]:
print("----- Test original -----\n")
for take in tab:
    model_path = "BestScientific_"+best_model[0]+".pkl"
    with open(model_path, "rb") as f:
        sentiment_model = pickle.load(f)

    print("Modèle chargé :", model_path)
    
    # Prédiction
    prediction = sentiment_model.predict([take])[0]

    print("Avis :", take)
    print("Prédiction :", prediction)
    print()

print("----- Nouveau test -----\n")
for take in tweets_task1:
    model_path = "BestScientific_"+best_model[0]+".pkl"
    with open(model_path, "rb") as f:
        sentiment_model = pickle.load(f)

    print("Modèle chargé :", model_path)
    
    # Prédiction
    prediction = sentiment_model.predict([take])[0]

    print("Avis :", take)
    print("Prédiction :", prediction)
    print()
    

----- Test original -----

Modèle chargé : BestScientific_SVC.pkl
Avis : Covid is chinese made
Prédiction : 1

Modèle chargé : BestScientific_SVC.pkl
Avis : Albatross is the best bird
Prédiction : 0

Modèle chargé : BestScientific_SVC.pkl
Avis : Buy Loreal for cheap
Prédiction : 0

Modèle chargé : BestScientific_SVC.pkl
Avis : the earth is flat
Prédiction : 0

Modèle chargé : BestScientific_SVC.pkl
Avis : Moon landing was fake
Prédiction : 1

Modèle chargé : BestScientific_SVC.pkl
Avis : the space doesn't exist
Prédiction : 0

----- Nouveau test -----

Modèle chargé : BestScientific_SVC.pkl
Avis : Poverty is the greatest cause behind child labour.
Prédiction : 1

Modèle chargé : BestScientific_SVC.pkl
Avis : Does daily routine help prevent bipolar disorder?
Prédiction : 1

Modèle chargé : BestScientific_SVC.pkl
Avis : Lupus Research Institute Awards $1M Grants to Discover What Causes Lupus
Prédiction : 1

Modèle chargé : BestScientific_SVC.pkl
Avis : The House science committee is now 

In [10]:
# voir si enlevez les emojis ou les garder c'est mieux
# voir quelle combinaison de prétraitements donne les meilleurs résultats
# vectoriser le texte

indices = np.where(best_model[1].predict(df.text) == 1)[0]
scientific_df = df.iloc[indices]

scientific_df_safe = df[df['science_related'] == 1].reset_index(drop=True)

print(f"Tweets prédits scientifiques (cascade inférence) : {len(scientific_df)}")
print(f"Tweets vraiment scientifiques (entraînement Stage 2/3) : {len(scientific_df_safe)}")

Tweets prédits scientifiques (cascade inférence) : 436
Tweets vraiment scientifiques (entraînement Stage 2/3) : 375


In [11]:
# Classification en 2 classes : {CLAIM,REF} vs. {CONTEXT}

X_2c = scientific_df_safe.text
y_2c = ((scientific_df_safe['scientific_claim'] == 1) | (scientific_df_safe['scientific_reference'] == 1)).astype(int)

# astype(int) : Transforme (True,False) en (1,0)


print("Distribution des classes :")
print(f"  Classe 0 (CONTEXT)    : {(y_2c == 0).sum()} tweets")
print(f"  Classe 1 (CLAIM ou REF): {(y_2c == 1).sum()} tweets")

X_train_2c, X_test_2c, y_train_2c, y_test_2c = train_test_split(
    X_2c, y_2c, test_size=0.2, stratify=y_2c, random_state=42
)

best_model_2c = findBestModel(X_train_2c, y_train_2c)
best_model_2c[1].fit(X_train_2c, y_train_2c)

print("\n=== Évaluation sur jeu de test indépendant (Stage 2) ===")
y_pred_2c = best_model_2c[1].predict(X_test_2c)
print(classification_report(y_test_2c, y_pred_2c, target_names=["CONTEXT", "CLAIM/REF"]))
print(f"Balanced accuracy (held-out): {metrics.balanced_accuracy_score(y_test_2c, y_pred_2c):.3f}")

# Re-fit final sur l'intégralité pour le pickle.
best_model_2c[1].fit(X_2c, y_2c)
print("Meilleur modèle :", best_model_2c[0])

model_path = "BestTwoClass_" + best_model_2c[0] + ".pkl"
with open(model_path, "wb") as f:
    pickle.dump(best_model_2c[1], f)
print("Modèle sauvegardé :", model_path)

[I 2026-05-02 22:39:08,609] A new study created in memory with name: no-name-7f2914c3-ace0-458c-8448-300c29ed3cb5


Distribution des classes :
  Classe 0 (CONTEXT)    : 33 tweets
  Classe 1 (CLAIM ou REF): 342 tweets


[I 2026-05-02 22:40:28,735] Trial 0 finished with value: 0.5 and parameters: {'model_name': 'RandomForest', 'sampling': 'oversample', 'norm_strategy': 'lemma', 'norm_lowercase': True, 'norm_stopwords': True, 'norm_digit': False, 'norm_emoji': True, 'vec_analyzer': 'char_wb', 'vec_ngram_char_min': 4, 'vec_ngram_char_max': 6, 'vec_min_df': 1, 'vec_max_df': 0.8784716024072278, 'vec_sublinear_tf': False, 'n_estimators': 89, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.5.
[I 2026-05-02 22:41:45,979] Trial 1 finished with value: 0.48778689863595515 and parameters: {'model_name': 'CoinToss', 'sampling': 'oversample', 'norm_strategy': 'stem', 'norm_lowercase': False, 'norm_stopwords': False, 'norm_digit': True, 'norm_emoji': False, 'vec_analyzer': 'char_wb', 'vec_ngram_char_min': 3, 'vec_ngram_char_max': 5, 'vec_min_df': 3, 'vec_max_df': 0.8372453300057505, 'vec_sublinear_tf': True}. Best is trial 0 with value: 0.5.
[I 2026-05-02 22:43:03,262] 

Meilleur score : 0.73
Meilleurs paramètres : {'model_name': 'DecisionTree', 'sampling': 'none', 'norm_strategy': 'none', 'norm_lowercase': False, 'norm_stopwords': True, 'norm_digit': True, 'norm_emoji': True, 'vec_analyzer': 'word', 'vec_ngram_word': '1_3', 'vec_min_df': 4, 'vec_max_df': 0.9124550780328005, 'vec_sublinear_tf': False, 'max_depth': 5, 'criterion': 'entropy'}

=== Évaluation sur jeu de test indépendant (Stage 2) ===
              precision    recall  f1-score   support

     CONTEXT       0.11      0.43      0.17         7
   CLAIM/REF       0.91      0.63      0.75        68

    accuracy                           0.61        75
   macro avg       0.51      0.53      0.46        75
weighted avg       0.84      0.61      0.69        75

Balanced accuracy (held-out): 0.530
Meilleur modèle : DecisionTree
Modèle sauvegardé : BestTwoClass_DecisionTree.pkl


In [12]:
model_path = "BestTwoClass_" + best_model_2c[0] + ".pkl"
with open(model_path, "rb") as f:
    model_2c = pickle.load(f)
print("Modèle chargé :", model_path)
print()

print("----- Test original -----\n")
for take in tab:
    prediction = model_2c.predict([take])[0]
    label = "CLAIM ou REF" if prediction == 1 else "CONTEXT"
    print("Tweet :", take)
    print(f"Prédiction : {prediction} ({label})")
    print()


print("----- Nouveau test -----\n")
for take in tweets_task2:
    prediction = model_2c.predict([take])[0]
    label = "CLAIM ou REF" if prediction == 1 else "CONTEXT"
    print("Tweet :", take)
    print(f"Prédiction : {prediction} ({label})")
    print()

Modèle chargé : BestTwoClass_DecisionTree.pkl

----- Test original -----

Tweet : Covid is chinese made
Prédiction : 1 (CLAIM ou REF)

Tweet : Albatross is the best bird
Prédiction : 1 (CLAIM ou REF)

Tweet : Buy Loreal for cheap
Prédiction : 1 (CLAIM ou REF)

Tweet : the earth is flat
Prédiction : 1 (CLAIM ou REF)

Tweet : Moon landing was fake
Prédiction : 1 (CLAIM ou REF)

Tweet : the space doesn't exist
Prédiction : 1 (CLAIM ou REF)

----- Nouveau test -----

Tweet : Poverty is the greatest cause behind child labour.
Prédiction : 1 (CLAIM ou REF)

Tweet : Does daily routine help prevent bipolar disorder?
Prédiction : 1 (CLAIM ou REF)

Tweet : Couch-lock highs lead to sleeping in the couch.
Prédiction : 1 (CLAIM ou REF)

Tweet : Can playing games improve lives? via @PostArcade
Prédiction : 1 (CLAIM ou REF)

Tweet : #MSGDonated4Million to support the research of Thalassemia & AIDS cure
Prédiction : 0 (CONTEXT)

Tweet : pls support #NIH funding to help fight #pancreaticcancer @PanCAN


In [ ]:
# Classification en 3 classes : {CLAIM} vs. {REF} vs. {CONTEXT}

def make_3class_label(row):
    if row['scientific_claim'] == 1:
        return 0  # CLAIM
    elif row['scientific_reference'] == 1:
        return 1  # REF
    else:
        return 2  # CONTEXT

y_3c = scientific_df_safe.apply(make_3class_label, axis=1)
X_3c = scientific_df_safe.text

print("Distribution des classes :")
print(f"  Classe 0 (CLAIM)  : {(y_3c == 0).sum()} tweets")
print(f"  Classe 1 (REF)    : {(y_3c == 1).sum()} tweets")
print(f"  Classe 2 (CONTEXT): {(y_3c == 2).sum()} tweets")

X_train_3c, X_test_3c, y_train_3c, y_test_3c = train_test_split(
    X_3c, y_3c, test_size=0.2, stratify=y_3c, random_state=42
)

best_model3 = findBestModel(X_train_3c, y_train_3c)
best_model3[1].fit(X_train_3c, y_train_3c)

print("\n=== Évaluation sur jeu de test indépendant (Stage 3) ===")
y_pred_3c = best_model3[1].predict(X_test_3c)
print(classification_report(y_test_3c, y_pred_3c, target_names=["CLAIM", "REF", "CONTEXT"]))
print(f"Balanced accuracy (held-out): {metrics.balanced_accuracy_score(y_test_3c, y_pred_3c):.3f}")

# Re-fit final sur l'intégralité pour le pickle.
best_model3[1].fit(X_3c, y_3c)
print("Meilleur modèle :", best_model3[0])

model_path = "BestThreeLabel_" + best_model3[0] + ".pkl"
with open(model_path, "wb") as f:
    pickle.dump(best_model3[1], f)
print("Modèle sauvegardé :", model_path)

[I 2026-05-02 23:34:24,731] A new study created in memory with name: no-name-fad4b7c4-02df-42a1-86ac-194504a5dc12


Distribution des classes :
  Classe 0 (CLAIM)  : 263 tweets
  Classe 1 (REF)    : 79 tweets
  Classe 2 (CONTEXT): 33 tweets


[I 2026-05-02 23:35:42,921] Trial 0 finished with value: 0.4242095227161216 and parameters: {'model_name': 'CoinToss', 'sampling': 'none', 'norm_strategy': 'none', 'norm_lowercase': True, 'norm_stopwords': True, 'norm_digit': True, 'norm_emoji': False, 'vec_analyzer': 'char_wb', 'vec_ngram_char_min': 2, 'vec_ngram_char_max': 4, 'vec_min_df': 5, 'vec_max_df': 0.9969596618398486, 'vec_sublinear_tf': True}. Best is trial 0 with value: 0.4242095227161216.
[I 2026-05-02 23:37:01,710] Trial 1 finished with value: 0.4750197347871767 and parameters: {'model_name': 'KNeighbors', 'sampling': 'none', 'norm_strategy': 'none', 'norm_lowercase': True, 'norm_stopwords': False, 'norm_digit': True, 'norm_emoji': False, 'vec_analyzer': 'word', 'vec_ngram_word': '1_2', 'vec_min_df': 2, 'vec_max_df': 0.8328539978975891, 'vec_sublinear_tf': True, 'n_neighbors': 9, 'weights': 'distance', 'p': 2}. Best is trial 1 with value: 0.4750197347871767.
[I 2026-05-02 23:38:18,813] Trial 2 finished with value: 0.39082

Meilleur score : 0.653
Meilleurs paramètres : {'model_name': 'SVC', 'sampling': 'none', 'norm_strategy': 'stem', 'norm_lowercase': False, 'norm_stopwords': True, 'norm_digit': True, 'norm_emoji': True, 'vec_analyzer': 'word', 'vec_ngram_word': '1_1', 'vec_min_df': 4, 'vec_max_df': 0.7904775600429366, 'vec_sublinear_tf': True, 'C': 9.985217066501045, 'kernel': 'linear', 'gamma': 'auto'}

=== Évaluation sur jeu de test indépendant (Stage 3) ===
              precision    recall  f1-score   support

       CLAIM       0.89      0.63      0.74        52
         REF       0.33      0.69      0.45        16
     CONTEXT       0.20      0.14      0.17         7

    accuracy                           0.60        75
   macro avg       0.48      0.49      0.45        75
weighted avg       0.71      0.60      0.63        75

Balanced accuracy (held-out): 0.488
Meilleur modèle : SVC
Modèle sauvegardé : BestThreeLabel_SVC.pkl


In [ ]:
class_names = {0: "CLAIM", 1: "REF", 2: "CONTEXT"}

model_path = "BestThreeLabel_" + best_model3[0] + ".pkl"
with open(model_path, "rb") as f:
    model_3c = pickle.load(f)
print("Modèle chargé :", model_path)
print()

print("----- Test original -----\n")
for take in tab:
    prediction = model_3c.predict([take])[0] 
    probabilities = model_3c.predict_proba([take])[0]

    print("Tweet :", take)
    print(f"Prédiction : {prediction} ({class_names[prediction]})")
    print("Probabilités par classe :")
    for class_idx, prob in enumerate(probabilities):
        print(f"  Classe {class_idx} ({class_names[class_idx]}): {prob:.4f}")
    print()

print("----- Nouveau test -----\n")
for take in tweets_task3:
    prediction = model_3c.predict([take])[0] 
    probabilities = model_3c.predict_proba([take])[0]

    print("Tweet :", take)
    print(f"Prédiction : {prediction} ({class_names[prediction]})")
    print("Probabilités par classe :")
    for class_idx, prob in enumerate(probabilities):
        print(f"  Classe {class_idx} ({class_names[class_idx]}): {prob:.4f}")
    print()

Modèle chargé : BestThreeLabel_SVC.pkl

----- Test original -----

Tweet : Covid is chinese made
Prédiction : 0 (CLAIM)
Probabilités par classe :
  Classe 0 (CLAIM): 0.8780
  Classe 1 (REF): 0.1055
  Classe 2 (CONTEXT): 0.0165

Tweet : Albatross is the best bird
Prédiction : 0 (CLAIM)
Probabilités par classe :
  Classe 0 (CLAIM): 0.8280
  Classe 1 (REF): 0.1398
  Classe 2 (CONTEXT): 0.0322

Tweet : Buy Loreal for cheap
Prédiction : 0 (CLAIM)
Probabilités par classe :
  Classe 0 (CLAIM): 0.8280
  Classe 1 (REF): 0.1398
  Classe 2 (CONTEXT): 0.0322

Tweet : the earth is flat
Prédiction : 0 (CLAIM)
Probabilités par classe :
  Classe 0 (CLAIM): 0.9189
  Classe 1 (REF): 0.0673
  Classe 2 (CONTEXT): 0.0137

Tweet : Moon landing was fake
Prédiction : 0 (CLAIM)
Probabilités par classe :
  Classe 0 (CLAIM): 0.8280
  Classe 1 (REF): 0.1398
  Classe 2 (CONTEXT): 0.0322

Tweet : the space doesn't exist
Prédiction : 0 (CLAIM)
Probabilités par classe :
  Classe 0 (CLAIM): 0.8280
  Classe 1 (REF): 0.